In [24]:
import pandas as pd
import os
import numpy as np
from configs.constants import MCBOOST_NAME

In [25]:
type = 'tuned'
# type = 'oob'
# res_dir = 'mc_industry_results/tuned/'
res_dir = f'mc_industry_results/{type}/'
res_files = os.listdir(res_dir)
csv_export_path = f'kdd_mcboost_results_{type}_all_datasets.csv'

In [26]:
dfs = []
for f in res_files:
    if not f.endswith('.pkl'):
        continue
    dfs.append(pd.read_pickle(os.path.join(res_dir, f)))
df = pd.concat(dfs)
df = df[df.seed == 15].copy()
df['mcb_algorithm'] = df.mcb_algorithm.fillna('base_model')

In [27]:
# Categorize the different mcboost variants based on parameter values
def categorize_mcboost(mcb_algorithm, params):

    if mcb_algorithm == 'HKRR':
        return f'HKRR_{params["lambda"]}_{params["alpha"]}'
    if mcb_algorithm != MCBOOST_NAME:
        return mcb_algorithm
    if mcb_algorithm == MCBOOST_NAME:
        return params['name']

    return None

In [28]:
resolved_algos = [categorize_mcboost(row['mcb_algorithm'], row['mcb_algorithm_params']) for _, row in df.iterrows()]

In [29]:
df['mcb_algorithm'] = resolved_algos

In [30]:
# There was a bug in the parms['name'] construction, fix:
df['mcb_algorithm'] = df.mcb_algorithm.replace({"20": MCBOOST_NAME + "_msh_20"})

In [31]:
df = df[df.mcb_algorithm.notnull()]

In [32]:
df_test = df[df.set_name == 'test']
### For HKRR take the validation results to find best params per dataset and remove other results from the test results
df_val_hkrr = df[(df.set_name == 'validation') & (df.mcb_algorithm.str.startswith('HKRR')) & (df.group == 'agg')].reset_index()
best_hkrr_params_per_dataset = df_val_hkrr.iloc[df_val_hkrr.groupby('dataset')['logloss'].idxmin()][['dataset', 'mcb_algorithm']].assign(matched=True)
df_test = pd.merge(df_test, best_hkrr_params_per_dataset, on=['dataset', 'mcb_algorithm'], how='left')
df_test['select'] = df_test.apply(lambda row: not np.isnan(row.matched) or not row.mcb_algorithm.startswith('HKRR'), axis=1)
df_test = df_test[df_test.select]
df_test = df_test.drop(columns=['matched', 'select'])
df_test['mcb_algorithm'] = df_test.mcb_algorithm.apply(lambda x: x if not x.startswith('HKRR') else 'HKRR')

In [33]:
# Check if we're left with one result per dataset per algorithm on aggregated test data
(df_test[df_test.group == 'agg'].groupby(['dataset', 'mcb_algorithm']).group.count() != 1).any()

np.False_

In [34]:
res_tab_index_cols = ['dataset', 'model', 'mcb_algorithm']
metric_name_mapping = {
    'MCE_perc': 'MCE_perc_features',
    'MCE_sigma': 'MCE_sigma_features',
    'ECCE_perc': 'global_ECCE_perc',
    'ECCE_sigma': 'global_ECCE_sigma',
    'ECE': 'global_ECE',
    'logloss': 'logloss',
    'prauc': 'prauc',
    'fit_time': 'fit_time',
    'num_rounds': 'num_rounds_mcboost',

}
agg_df = df_test[df_test.group == 'agg']

res_tab = agg_df[res_tab_index_cols + list(metric_name_mapping.keys()) + ['mcb_algorithm_params']].set_index(res_tab_index_cols)
res_tab = res_tab.rename(columns=metric_name_mapping)

max_df = df_test[df_test.group == 'max']
max_df_sel = max_df[res_tab_index_cols + ['ECCE_perc', 'ECCE_sigma', 'smECE']].set_index(res_tab_index_cols)
max_df_sel = max_df_sel.rename(columns={'ECCE_perc': 'MCE_perc_groups', 'ECCE_sigma': 'MCE_sigma_groups', 'smECE': 'max_group_smECE'})
res_tab = res_tab.join(max_df_sel, how='left')

In [35]:
# Check there are no extra rows
(res_tab.reset_index(drop=False).groupby(['dataset', 'mcb_algorithm']).prauc.count() != 1).any()

np.False_

In [36]:
# Take just LogisticRegression
lr_res_tab = res_tab.xs('LogisticRegression', level=1)

In [37]:
pretty_colnames = {
    'global_ECCE_perc': 'ECCE-%',
    'global_ECCE_sigma': 'ECCE-σ',
    'global_ECE': 'ECE',
    'logloss': 'Log-Loss',
    'prauc': 'PRAUC',
    'MCE_perc_features': 'MCE-% (all)',
    'MCE_sigma_features': 'MCE-σ (all)' ,
    'MCE_perc_groups': 'MCE-% (grp)',
    'MCE_sigma_groups': 'MCE-σ (grp)',
    'max_group_smECE': 'max smECE (grp)',
}

In [38]:
# temporary rename
# lr_res_tab.reset_index(drop=False, inplace=True)
# lr_res_tab = lr_res_tab.rename({'CASMCBoost_msh_0_ablation': 'CASMCBoost_msh_20', 'CASMCBoost_unshrink_ablation': 'CASMCBoost_no_unshrink', 'CAS_MCBoost_one_round_ablation': 'CASMCBoost_one_round'})
# lr_res_tab.set_index(['dataset', 'mcb_algorithm'], inplace=True)

In [39]:
lr_res_tab.to_csv(csv_export_path, index=True)

In [40]:
global_metrics_res_tab = lr_res_tab[['global_ECCE_perc', 'global_ECCE_sigma', 'global_ECE', 'logloss', 'prauc']].rename(columns=pretty_colnames)
mc_metrics_res_tab = lr_res_tab[['MCE_perc_features', 'MCE_sigma_features', 'MCE_perc_groups', 'MCE_sigma_groups', 'max_group_smECE']].rename(columns=pretty_colnames)

### Latex tables

In [41]:
import pandas as pd
import re

def escape_latex(text):
    """Escape LaTeX special characters in text."""
    if not isinstance(text, str):
        return text
    replacements = {
        '\\': r'\textbackslash{}',
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
    }
    pattern = re.compile('|'.join(re.escape(k) for k in replacements))
    return pattern.sub(lambda m: replacements[m.group()], text)

def multiindex_to_latex_highlighted(df, digits=4):
    """Convert MultiIndex DataFrame to LaTeX tabular format with per-group highlights and lines after each top-level group."""
    higher_is_better = ['PRAUC']
    numeric_cols = df.select_dtypes(include='number').columns

    group_level = 0
    group_min, group_max = {}, {}

    for name, group in df.groupby(level=group_level):
        for col in numeric_cols:
            if col in higher_is_better:
                group_max[(name, col)] = group[col].max()
            else:
                group_min[(name, col)] = group[col].min()

    latex_lines = []
    last_values = [""] * df.index.nlevels

    headers = [escape_latex(h) for h in list(df.index.names) + list(df.columns)]
    latex_lines.append("\\begin{tabular}{" + "l" * len(headers) + "}")
    latex_lines.append("\\toprule")
    latex_lines.append(" & ".join(headers) + " \\\\")
    latex_lines.append("\\midrule")

    current_group = None
    for idx, row in df.iterrows():
        top_group = idx[group_level]

        # If a new group starts, insert a line before it (but not before the first group)
        if current_group is not None and top_group != current_group:
            latex_lines.append("\\midrule")
        current_group = top_group

        row_items = []
        for level, value in enumerate(idx):
            if value == last_values[level]:
                row_items.append("")
            else:
                row_items.append(escape_latex(str(value)))
                last_values[level] = value
            for reset_level in range(level + 1, df.index.nlevels):
                last_values[reset_level] = ""

        for col in df.columns:
            val = row[col]
            if isinstance(val, float):
                formatted = f"{val:.{digits}f}"
                if col in higher_is_better and val == group_max.get((top_group, col), None):
                    formatted = f"\\textbf{{{formatted}}}"
                elif col not in higher_is_better and val == group_min.get((top_group, col), None):
                    formatted = f"\\textbf{{{formatted}}}"
            else:
                formatted = escape_latex(str(val))
            row_items.append(formatted)

        latex_lines.append(" & ".join(row_items) + " \\\\")

    latex_lines.append("\\bottomrule")
    latex_lines.append("\\end{tabular}")
    return "\n".join(latex_lines)


In [42]:
print(multiindex_to_latex_highlighted(mc_metrics_res_tab))

\begin{tabular}{lllllll}
\toprule
dataset & mcb\_algorithm & MCE-\% (all) & MCE-σ (all) & MCE-\% (grp) & MCE-σ (grp) & max smECE (grp) \\
\midrule
MEPS & base\_model & 15.2178 & 3.9940 & 45.7745 & 3.2849 & 0.0819 \\
 & MCGrad\_msh\_0 & 14.8103 & 3.9864 & 39.8124 & 3.3902 & 0.0740 \\
 & MCGrad & 15.1974 & 4.0779 & 38.6883 & 3.3449 & 0.0773 \\
 & MCGrad\_group\_features & 14.4755 & 3.8291 & 45.9970 & 3.3628 & 0.0734 \\
 & MCGrad\_no\_unshrink & 14.4527 & 3.8059 & 40.2162 & \textbf{3.0839} & 0.0761 \\
 & MCGrad\_one\_round & 14.8665 & 4.0039 & 36.8738 & 3.4017 & 0.0769 \\
 & DFMC & 13.9009 & 3.6537 & 44.8676 & 3.4119 & 0.0737 \\
 & HKRR & 18.2990 & 4.8624 & \textbf{32.4973} & 3.1814 & 0.1054 \\
 & Isotonic & \textbf{13.6096} & \textbf{3.6076} & 43.6074 & 3.4070 & \textbf{0.0713} \\
\midrule
acs\_public\_health\_insurance\_all\_states & base\_model & 11.4533 & 38.5160 & 23.9771 & 22.8242 & 0.1043 \\
 & MCGrad\_msh\_0 & 3.6553 & 13.4951 & 8.2610 & 8.5799 & 0.0397 \\
 & MCGrad & 2.5336 & 9.3

In [43]:
print(multiindex_to_latex_highlighted(global_metrics_res_tab))

\begin{tabular}{lllllll}
\toprule
dataset & mcb\_algorithm & ECCE-\% & ECCE-σ & ECE & Log-Loss & PRAUC \\
\midrule
MEPS & base\_model & 9.0583 & 2.3774 & 0.0215 & 0.3177 & 0.6211 \\
 & MCGrad\_msh\_0 & 9.0082 & 2.4247 & 0.0179 & \textbf{0.3137} & \textbf{0.6279} \\
 & MCGrad & 9.2764 & 2.4891 & 0.0174 & 0.3155 & 0.6217 \\
 & MCGrad\_group\_features & 8.2807 & 2.1904 & 0.0177 & 0.3161 & 0.6209 \\
 & MCGrad\_no\_unshrink & 7.3920 & 1.9466 & 0.0229 & 0.3157 & 0.6213 \\
 & MCGrad\_one\_round & 8.5643 & 2.3066 & 0.0209 & 0.3150 & 0.6225 \\
 & DFMC & 7.1199 & 1.8714 & 0.0176 & 0.3157 & 0.6195 \\
 & HKRR & 9.5242 & 2.5308 & 0.0252 & 0.3367 & 0.5317 \\
 & Isotonic & \textbf{6.4659} & \textbf{1.7140} & \textbf{0.0118} & 0.3160 & 0.5945 \\
\midrule
acs\_public\_health\_insurance\_all\_states & base\_model & 3.4768 & 11.6919 & 0.0181 & 0.5186 & 0.5969 \\
 & MCGrad\_msh\_0 & 3.2664 & 12.0593 & 0.0104 & 0.4557 & 0.6875 \\
 & MCGrad & 2.4538 & 9.1011 & 0.0128 & 0.4531 & 0.6918 \\
 & MCGrad\_group\_f

In [44]:
d = lr_res_tab.reset_index()

In [45]:
d[d.dataset.str.startswith('acs_mobil')]

,dataset,mcb_algorithm,MCE_perc_features,MCE_sigma_features,global_ECCE_perc,global_ECCE_sigma,global_ECE,logloss,prauc,fit_time,num_rounds_mcboost,mcb_algorithm_params,MCE_perc_groups,MCE_sigma_groups,max_group_smECE
63,acs_mobility_all_states,base_model,24.6077,53.287800,3.1376,6.7944,0.0189,0.5539,0.8061,NaN,NaN,None,8.2969,8.7336,0.0646
64,acs_mobility_all_states,MCGrad_msh_0,4.0147,9.345300,4.0147,9.3453,0.0110,0.5077,0.8698,1269.336655,1.0,"{'name': 'MCGrad_msh_0', 'feature_type': 'feat...",2.8399,6.0279,0.0320
65,acs_mobility_all_states,MCGrad,1.9569,4.505800,1.7376,4.0008,0.0051,0.5055,0.8724,1187.573816,1.0,"{'name': 'MCGrad', 'feature_type': 'features',...",2.3132,2.9779,0.0280
66,acs_mobility_all_states,MCGrad_group_features,25.0014,54.396801,0.7819,1.7013,0.0023,0.5505,0.8116,3132.107956,4.0,"{'name': 'MCGrad_group_features', 'feature_typ...",2.2560,2.0215,0.0231
67,acs_mobility_all_states,MCGrad_no_unshrink,1.9357,4.436400,0.9358,2.1447,0.0049,0.5065,0.8696,1256.193471,1.0,"{'name': 'MCGrad_no_unshrink', 'feature_type':...",2.7206,2.0510,0.0263
68,acs_mobility_all_states,MCGrad_one_round,2.1302,4.920000,2.1302,4.9200,0.0067,0.5052,0.8726,381.369883,1.0,"{'name': 'MCGrad_one_round', 'feature_type': '...",2.6100,3.7256,0.0296
69,acs_mobility_all_states,DFMC,24.9913,54.342800,0.7858,1.7087,0.0027,0.5503,0.8119,336.658073,1.0,"{'name': 'DFMC', 'feature_type': 'group', 'uns...",2.1457,2.1307,0.0229
70,acs_mobility_all_states,HKRR,25.0864,54.124199,1.7354,3.7442,0.0072,0.5537,0.7916,8.745392,NaN,"{'lambda': 0.1, 'alpha': 0.0125}",4.5088,5.0506,0.0425
71,acs_mobility_all_states,Isotonic,25.1602,54.640499,0.6945,1.5082,0.0015,0.5520,0.8046,0.078395,NaN,{},8.0610,5.5356,0.0678


In [46]:
len(d)

99